In [3]:
# Imports, as always...
import os
import numpy as np
import pandas as pd
from typing import List
from tqdm import tqdm

# Circuitry.
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

# Braket.
from qiskit_braket_provider import BraketProvider
from braket.tracking import Tracker
from braket.aws.aws_quantum_task import AwsQuantumTask
from qiskit.qasm3 import loads

# Ignore warnings.
import warnings
warnings.filterwarnings('ignore')

# Depth-Varied Experiments

In [4]:
# Function to generate a circuit using the brickwork architecture.
def generate_brickwork_circuit(n : int, d : int, rng = None):
    # A brickwork circuit only really makes sense with at least 3 qubits.
    assert n > 2, 'Brickwork architectures require at least 3 qubits to be worth the bother.'

    # Instantiate a circuit object.
    circuit = QuantumCircuit(n)

    # Iteratively adding layers of bricks.
    for l in range(d):
        # Even layer arrangement.
        if l % 2 == 0:
            for i in range(n // 2):
                # Sample a Haar random unitary from U(4).
                circuit.append(
                    UnitaryGate(random_unitary(4, rng), label='$U_{' + str(l) + str(i) + '}$'),
                    [2*i, 2*i + 1]
                )

        # Odd layer arrangement.
        else:
            for i in range(1, n //2 + 1):
                # And again.
                circuit.append(
                    UnitaryGate(random_unitary(4, rng), label='$U_{' + str(l) + str(i) + '}$'),
                    [2*i - 1, 2*i]
                )

    return circuit

# Demo.
demo_circuit = generate_brickwork_circuit(n=5, d=4, rng=np.random.default_rng(seed=42))
demo_circuit.measure_all()
demo_circuit.draw()

┌───────────┐             ┌───────────┐              ░ ┌─┐            
   q_0: ┤0          ├─────────────┤0          ├──────────────░─┤M├────────────
        │  $U_{00}$ │┌───────────┐│  $U_{20}$ │┌───────────┐ ░ └╥┘┌─┐         
   q_1: ┤1          ├┤0          ├┤1          ├┤0          ├─░──╫─┤M├─────────
        ├───────────┤│  $U_{11}$ │├───────────┤│  $U_{31}$ │ ░  ║ └╥┘┌─┐      
   q_2: ┤0          ├┤1          ├┤0          ├┤1          ├─░──╫──╫─┤M├──────
        │  $U_{01}$ │├───────────┤│  $U_{21}$ │├───────────┤ ░  ║  ║ └╥┘┌─┐   
   q_3: ┤1          ├┤0          ├┤1          ├┤0          ├─░──╫──╫──╫─┤M├───
        └───────────┘│  $U_{12}$ │└───────────┘│  $U_{32}$ │ ░  ║  ║  ║ └╥┘┌─┐
   q_4: ─────────────┤1          ├─────────────┤1          ├─░──╫──╫──╫──╫─┤M├
                     └───────────┘             └───────────┘ ░  ║  ║  ║  ║ └╥┘
meas: 5/════════════════════════════════════════════════════════╩══╩══╩══╩══╩═
                                                                0  1  2  3  4

In [ ]:
# Set region.
# IonQ and QuEra are on us-east-1, IQM is on eu-north-1, and Rigetti is on us-west-1.
os.environ['AWS_REGION'] = 'us-east-1'

# Getting a device.
provider = BraketProvider()
qpu = provider.get_backend('Garnet')
print(qpu)

In [ ]:
# Generating data.
def depth_varied_routine(
    n: int,
    ds: List[int],
    n_circuits: int,
    device,
    n_shots: int = 128,
    save_path = None,
    verbose: bool = False
) -> pd.DataFrame:
    # Dataframe to hold metadata.
    metadata_df = pd.DataFrame(columns=['n', 'd', 'circuit_seed', 'n_shots', 'device', 'task_id'])
    if save_path:
        metadata_df.to_csv(save_path, index=False) # Writing headers.

    # Start tracking costs.
    t = Tracker().start()

    seed = 0
    for d in (tqdm(ds, desc='Depths') if verbose else ds):
        for _ in range(n_circuits):
            # Next seed.
            seed += 1

            # Build the experiment circuit.
            experiment_circuit = generate_brickwork_circuit(n, d, np.random.default_rng(seed))
            experiment_circuit.measure_all()

            # Queue (and run) task.
            qpu_task = device.run(experiment_circuit, shots=n_shots)

            # Store metadata.
            run_metadata_df = pd.DataFrame({
                'n' : [n],
                'd' : [d],
                'circuit_seed' : [seed],
                'n_shots' : [n_shots],
                'device' : [device.name],
                'task_id' : [qpu_task.task_id()]
            })

            # Add to the overall results.
            metadata_df = pd.concat([metadata_df, run_metadata_df], ignore_index=True)

            # Save as we go (appending to file).
            if save_path:
                run_metadata_df.to_csv(save_path, mode='a', header=False, index=False)

    # Calculate costs.
    print(f'Total estimated cost: {t.qpu_tasks_cost() + t.simulator_tasks_cost():.3f} USD')

    # Return metadata (results to be collected from AWS using task_id).
    return metadata_df

In [ ]:
metadata_df = depth_varied_routine(
    n=5,
    ds=[5, 10, 15, 20, 25],
    n_circuits=500,
    device=qpu,
    n_shots=512,
    save_path='./data/depth-varied/Garnet.csv',
    verbose=True
)

In [ ]:
# For some inexplicable reason, Aria-1 and Ankaa-3 don't follow the same naming format.
# So change that manually...
if False:
    metadata_df.replace({'Aria 1' : 'Aria-1'}, inplace=True)
    metadata_df.to_csv('data/depth-varied/Aria-1.csv', index=False)

In [ ]:
# Reading in data, using the task_id column of the given dataframe.
def read_data_to_frame(
    metadata_df: pd.DataFrame
) -> pd.DataFrame:
    # Create copy.
    df = metadata_df.copy()

    # Assuming a fixed n -- I think it's cleaner.
    n = int(df.iloc[0]['n'])

    # Keeping a dataframe for measurement outcome probabilities.
    df[['x' + format(i, f'0{n}b') for i in range(2 ** n)]] = 0

    for i in tqdm(range(len(metadata_df))):
        # Get the task ID.
        task_id = metadata_df.iloc[i]['task_id']

        # Retrieve the task from AWS.
        retrieved_task = AwsQuantumTask(arn=task_id)

        # Translate the entries of the outcome distribution into the DataFrame.
        for state, prob in retrieved_task.result().measurement_probabilities.items():
            df.loc[i, 'x' + str(state)] = prob

    return df

In [ ]:
data_df = read_data_to_frame(metadata_df)

In [ ]:
# Things we'll need because Qiskit shits the bed around QASM 3.0...
header = 'OPENQASM 3.0;'
inclusion = 'include "stdgates.inc";'
xx_gate_definition = '''gate xx(theta) q0, q1 {
    h q0;
    h q1;
    cx q0, q1;
    rz(theta) q1;
    cx q0, q1;
    h q0;
    h q1;
}'''

In [ ]:
def add_circuit_qasm_to_df(df):
    qasms = []
    for i in tqdm(range(len(df))):
        task_id = df.iloc[i]['task_id']
        retrieved_task = AwsQuantumTask(arn=task_id)
        circuit_qasm = retrieved_task.result().additional_metadata.action.source.replace(header, f'{header}\n{inclusion}\n{xx_gate_definition}')
        circuit_qasm = circuit_qasm.replace('cnot', 'cx') # Lord knows why this difference exists AND needs correcting.
        qasms.append(circuit_qasm)
        
    df['QASM3'] = qasms
    return df

In [ ]:
data_df = add_circuit_qasm_to_df(data_df)

In [ ]:
# Computing the expected outcome distributions.
def compute_true_dists(
    data_df: pd.DataFrame
) -> pd.DataFrame:
    # Create copy.
    df = data_df.copy()

    # Assuming a fixed n -- I think it's cleaner.
    n = int(df.iloc[0]['n'])
    
    # Defining some more gates (amazingly, only Garnet uses these!)
    defs = '''gate phaseshift(lambda) q {
U(0, 0, lambda) q;
}
gate v q {
U(pi/2, -pi/2, pi/2) q;
}
gate rxx(theta) a, b {
    h a;
    cx a, b;
    phaseshift(theta) b;
    cx a, b;
    h a;
}
gate ryy(theta) a, b {
    rx(pi/2) a;
    rx(pi/2) b;
    cx a, b;
    phaseshift(theta) b;
    cx a, b;
    rx(-pi/2) a;
    rx(-pi/2) b;
}
gate iswap a, b {
    rxx(pi/2) a, b;
    ryy(pi/2) a, b;
}
'''

    # Keeping a dataframe for true outcome probabilities.
    df[['y' + format(i, f'0{n}b') for i in range(2 ** n)]] = 0
    
    for i in tqdm(range(len(data_df))):
        # Get the circuit.
        circuit_qasm = data_df.iloc[i]['QASM3']
        circuit_qasm = circuit_qasm.replace('OPENQASM 3.0;\ninclude "stdgates.inc";', f'OPENQASM 3.0;\ninclude "stdgates.inc";{defs}')
        circuit = loads(circuit_qasm)
        
        # Get the probabilities.
        circuit.remove_final_measurements()
        probabilities = Statevector(circuit).probabilities()
        
        # Translate to the data frame.
        for state, prob in zip(['y' + format(i, f'0{n}b') for i in range(2 ** n)], probabilities):
            df.loc[i, state] = prob
            
    return df

In [ ]:
data_df = compute_true_dists(data_df)

In [ ]:
# Compute discrepancy in output distributions.
def compute_discrepancies(
    data_df: pd.DataFrame
) -> pd.DataFrame:
    # Create copy.
    df = data_df.copy()

    # Assuming a fixed n -- I think it's cleaner.
    n = int(df.iloc[0]['n'])

    # Compute the discrepancies.
    states = [format(i, f'0{n}b') for i in range(2 ** n)]
    for state in tqdm(states):
        df[state] = df['x' + state] - df['y' + state]
            
    # Drop the excess columns.
    df.drop(columns=['x'+state for state in states]+['y'+state for state in states], inplace=True)
            
    return df

In [ ]:
data_df = compute_discrepancies(data_df)

In [ ]:
data_df.to_csv('./data/depth-varied/Garnet.csv')

# Time-Varied Experiments

In [5]:
def ghz_state_circuit(n : int):
    assert n > 1, "Come on now, don't be silly. At least 2 qubits."
    
    # Instantiate a circuit object.
    circuit = QuantumCircuit(n)
    
    # Hadamard.
    circuit.h(0)

    # CNOTs.
    for i in range(1, n):
        circuit.cx(0, i)

    # Measure.
    circuit.measure_all()
    
    return circuit

# Demo.
demo_circuit = ghz_state_circuit(n=5)
demo_circuit.draw()

┌───┐                     ░ ┌─┐            
   q_0: ┤ H ├──■────■────■────■───░─┤M├────────────
        └───┘┌─┴─┐  │    │    │   ░ └╥┘┌─┐         
   q_1: ─────┤ X ├──┼────┼────┼───░──╫─┤M├─────────
             └───┘┌─┴─┐  │    │   ░  ║ └╥┘┌─┐      
   q_2: ──────────┤ X ├──┼────┼───░──╫──╫─┤M├──────
                  └───┘┌─┴─┐  │   ░  ║  ║ └╥┘┌─┐   
   q_3: ───────────────┤ X ├──┼───░──╫──╫──╫─┤M├───
                       └───┘┌─┴─┐ ░  ║  ║  ║ └╥┘┌─┐
   q_4: ────────────────────┤ X ├─░──╫──╫──╫──╫─┤M├
                            └───┘ ░  ║  ║  ║  ║ └╥┘
meas: 5/═════════════════════════════╩══╩══╩══╩══╩═
                                     0  1  2  3  4

In [ ]:
# Set region.
os.environ['AWS_REGION'] = 'us-east-1'

# Getting a device.
provider = BraketProvider()
qpu = provider.get_backend('Garnet')
print(qpu)

In [ ]:
def time_varied_routine(
    n : int,
    steps: int,
    batch: int,
    device,
    n_shots: int = 128,
    save_path = None,
    verbose: bool = False
) -> pd.DataFrame:
    # Dataframe to hold metadata.
    metadata_df = pd.DataFrame(columns=['n', 'step', 'batch', 'n_shots', 'device', 'task_id'])
    if save_path:
        metadata_df.to_csv(save_path, index=False) # Writing headers.

    # Start tracking costs.
    t = Tracker().start()

    for step in (tqdm(range(1, steps+1), desc='Steps') if verbose else range(1, steps+1)):
        # Build the experiment circuit.
        experiment_circuit = ghz_state_circuit(n)

        # Queue (and run) task.
        qpu_task = device.run(experiment_circuit, shots=n_shots)

        # Store metadata.
        run_metadata_df = pd.DataFrame({
            'n' : [n],
            'step' : [step],
            'batch' : [batch],
            'n_shots' : [n_shots],
            'device' : [device.name],
            'task_id' : [qpu_task.task_id()]
        })

        # Add to the overall results.
        metadata_df = pd.concat([metadata_df, run_metadata_df], ignore_index=True)

        # Save as we go (appending to file).
        if save_path:
            run_metadata_df.to_csv(save_path, mode='a', header=False, index=False)

    # Calculate costs.
    print(f'Total estimated cost: {t.qpu_tasks_cost() + t.simulator_tasks_cost():.3f} USD')

    # Return metadata (results to be collected from AWS using task_id).
    return metadata_df

In [ ]:
df = time_varied_routine(n=5, steps=3, batch=1, device=qpu, verbose=True)